# Results analysis


##  Average throughput and latency computations

In [ ]:
from crayfish_utils import get_measurements

measurements = get_measurements("../results")

Compute average measurements per experiment.

In [ ]:
from collections import defaultdict
import copy
import pprint
from statistics import mean

average_measurements = defaultdict(dict)

for exp_footprint in measurements:
    for stream_processor in measurements[exp_footprint]:
        for model_name in measurements[exp_footprint][stream_processor]:
            for model_format in measurements[exp_footprint][stream_processor][model_name]: 
                if exp_footprint not in average_measurements:
                    average_measurements[exp_footprint] = {}
                if stream_processor not in average_measurements[exp_footprint]:
                    average_measurements[exp_footprint][stream_processor] = {}
                if model_name not in average_measurements[exp_footprint][stream_processor]:
                    average_measurements[exp_footprint][stream_processor][model_name] = {}
                if model_format not in average_measurements[exp_footprint][stream_processor][model_name]:
                    average_measurements[exp_footprint][stream_processor][model_name][model_format] = {"throughput": [], "latency": []}
                           
                measurement = measurements[exp_footprint][stream_processor][model_name][model_format]
                for mm in measurement:
                    throughput = mm['throughput']
                    latency = mm['avg_latency']
                    average_measurements[exp_footprint][stream_processor][model_name][model_format]["throughput"].append(throughput)
                    average_measurements[exp_footprint][stream_processor][model_name][model_format]["latency"].append(latency)

final_measurements = copy.deepcopy(average_measurements)
for exp in average_measurements.keys():
    print(exp)
    for stream_processor in average_measurements[exp].keys():
        for model_name in average_measurements[exp][stream_processor]:
            for model_format in average_measurements[exp][stream_processor][model_name]:
                throughputs = average_measurements[exp][stream_processor][model_name][model_format]["throughput"]
                latencies = average_measurements[exp][stream_processor][model_name][model_format]["latency"]
                print(stream_processor, "-", model_name, "-", model_format)
                print("  throughputs:", throughputs, "\n  avg:", mean(throughputs))
                print("  latencies:", latencies, "\n  avg:", mean(latencies))
                final_measurements[exp][stream_processor][model_name][model_format]["throughput"] = mean(throughputs)
                final_measurements[exp][stream_processor][model_name][model_format]["latency"] = mean(latencies)# * 0.001
    print()

pprint.pprint(final_measurements)


# Comparison between serving alternatives

### Sustainable Throughput

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl
from itertools import cycle
import itertools
import seaborn as sns
import random 

##### WindFlow e Flink

In [ ]:
th_labels = ["512_1_1", "2048_1_1", "4096_1_1"]
input_rates = [512, 2048, 4096]

wf_sp = "windflow"
flink_sp = "flink"

# Accumulatori throughput (WF)
wf_onnx_thr = []
wf_tfsm_thr = []
wf_tfsrv_thr = []
wf_torchsrv_thr = []

# Accumulatori throughput (Flink)
flink_onnx_thr = []
flink_tfsm_thr = []
flink_tfsrv_thr = []
flink_torchsrv_thr = []

for label in th_labels:
    th_exps = final_measurements.get(label, {})

    # --- WindFlow: estrazione throughput FFNN per backend ---
    if wf_sp in th_exps and "ffnn" in th_exps[wf_sp]:
        wf_ffnn = th_exps[wf_sp]["ffnn"]
        wf_onnx_thr.append(wf_ffnn.get("onnx", {}).get("throughput"))
        wf_tfsm_thr.append(wf_ffnn.get("tf-savedmodel", {}).get("throughput"))
        wf_tfsrv_thr.append(wf_ffnn.get("tf-serving", {}).get("throughput"))
        wf_torchsrv_thr.append(wf_ffnn.get("torchserve", {}).get("throughput"))
    else:
        wf_onnx_thr.append(None)
        wf_tfsm_thr.append(None)
        wf_tfsrv_thr.append(None)
        wf_torchsrv_thr.append(None)

    # --- Flink: estrazione throughput FFNN per backend ---
    if flink_sp in th_exps and "ffnn" in th_exps[flink_sp]:
        fl_ffnn = th_exps[flink_sp]["ffnn"]
        flink_onnx_thr.append(fl_ffnn.get("onnx", {}).get("throughput"))
        flink_tfsm_thr.append(fl_ffnn.get("tf-savedmodel", {}).get("throughput"))
        flink_tfsrv_thr.append(fl_ffnn.get("tf-serving", {}).get("throughput"))
        flink_torchsrv_thr.append(fl_ffnn.get("torchserve", {}).get("throughput"))
    else:
        flink_onnx_thr.append(None)
        flink_tfsm_thr.append(None)
        flink_tfsrv_thr.append(None)
        flink_torchsrv_thr.append(None)

def safe_max(xs):
    # max ignorando i None
    vals = [x for x in xs if x is not None]
    return max(vals) if vals else None

print("Sustainable throughput achieved [FFNN, N-N-N setup]")
print(f"WindFlow ONNX:          {safe_max(wf_onnx_thr)}")
print(f"WindFlow TF-SavedModel: {safe_max(wf_tfsm_thr)}")
print(f"WindFlow TF-Serving:    {safe_max(wf_tfsrv_thr)}")
print(f"WindFlow TorchServe:    {safe_max(wf_torchsrv_thr)}")
print("-" * 30)
print(f"Flink ONNX:             {safe_max(flink_onnx_thr)}")
print(f"Flink TF-SavedModel:    {safe_max(flink_tfsm_thr)}")
print(f"Flink TF-Serving:       {safe_max(flink_tfsrv_thr)}")
print(f"Flink TorchServe:       {safe_max(flink_torchsrv_thr)}")

print("\nMaximum Throughput per input rate (req/s)")
for i, r in enumerate(input_rates):
    print(
        f"ir = {r:>4}\n"
        f"   WF onnx = {wf_onnx_thr[i]} | tf-sm = {wf_tfsm_thr[i]} | tf-srv = {wf_tfsrv_thr[i]} | tsrv = {wf_torchsrv_thr[i]}\n"
        f"   FL onnx = {flink_onnx_thr[i]} | tf-sm = {flink_tfsm_thr[i]} | tf-srv = {flink_tfsrv_thr[i]} | tsrv = {flink_torchsrv_thr[i]}"
    )

In [ ]:
th_labels = ["512_1_1", "2048_1_1", "4096_1_1"]
input_rates = [512, 2048, 4096]

wf_sp = "windflow-32-N-32"
flink_sp = "flink-32-N-32"

# Accumulatori throughput (WF/Flink) per ONNX e TF-SavedModel
wf_onnx_thr = []
wf_tfsm_thr = []
flink_onnx_thr = []
flink_tfsm_thr = []

for label in th_labels:
    th_exps = final_measurements.get(label, {})

    # --- WindFlow: FFNN throughput ---
    if wf_sp in th_exps and "ffnn" in th_exps[wf_sp]:
        wf_onnx_thr.append(th_exps[wf_sp]["ffnn"].get("onnx", {}).get("throughput"))
        wf_tfsm_thr.append(th_exps[wf_sp]["ffnn"].get("tf-savedmodel", {}).get("throughput"))
    else:
        wf_onnx_thr.append(None)
        wf_tfsm_thr.append(None)

    # --- Flink: FFNN throughput ---
    if flink_sp in th_exps and "ffnn" in th_exps[flink_sp]:
        flink_onnx_thr.append(th_exps[flink_sp]["ffnn"].get("onnx", {}).get("throughput"))
        flink_tfsm_thr.append(th_exps[flink_sp]["ffnn"].get("tf-savedmodel", {}).get("throughput"))
    else:
        flink_onnx_thr.append(None)
        flink_tfsm_thr.append(None)

def safe_max(xs):
    # max ignorando i None
    vals = [x for x in xs if x is not None]
    return max(vals) if vals else None

print("Sustainable throughput achieved [FFNN, 32-1-32 setup]")
print(f"WindFlow ONNX:          {safe_max(wf_onnx_thr)}")
print(f"WindFlow TF-SavedModel: {safe_max(wf_tfsm_thr)}")
print("-" * 30)
print(f"Flink ONNX:             {safe_max(flink_onnx_thr)}")
print(f"Flink TF-SavedModel:    {safe_max(flink_tfsm_thr)}")

print("\nMax Throughput per input rate (req/s)")
for i, r in enumerate(input_rates):
    print(
        f"ir = {r:>4}\n"
        f"   WF onnx={wf_onnx_thr[i]} | tf-sm={wf_tfsm_thr[i]}\n"
        f"   FL onnx={flink_onnx_thr[i]} | tf-sm={flink_tfsm_thr[i]}"
    )

In [ ]:
th_labels = ["128_1_1"]
input_rates = [128]

wf_sp = "windflow"
flink_sp = "flink"

# Throughput WindFlow
wf_onnx_thr = []
wf_tfsm_thr = []
wf_tfsrv_thr = []
wf_torchsrv_thr = []

# Throughput Flink
flink_onnx_thr = []
flink_tfsm_thr = []
flink_tfsrv_thr = []
flink_torchsrv_thr = []

for label in th_labels:
    th_exps = final_measurements.get(label, {})

    # --- WindFlow / ResNet50 ---
    wf_res = th_exps.get(wf_sp, {}).get("resnet50", {})
    wf_onnx_thr.append(wf_res.get("onnx", {}).get("throughput"))
    wf_tfsm_thr.append(wf_res.get("tf-savedmodel", {}).get("throughput"))
    wf_tfsrv_thr.append(wf_res.get("tf-serving", {}).get("throughput"))
    wf_torchsrv_thr.append(wf_res.get("torchserve", {}).get("throughput"))

    # --- Flink / ResNet50 ---
    fl_res = th_exps.get(flink_sp, {}).get("resnet50", {})
    flink_onnx_thr.append(fl_res.get("onnx", {}).get("throughput"))
    flink_tfsm_thr.append(fl_res.get("tf-savedmodel", {}).get("throughput"))
    flink_tfsrv_thr.append(fl_res.get("tf-serving", {}).get("throughput"))
    flink_torchsrv_thr.append(fl_res.get("torchserve", {}).get("throughput"))

def safe_max(xs):
    # max ignorando i None
    vals = [x for x in xs if x is not None]
    return max(vals) if vals else None

print("Sustainable throughput achieved [ResNet50, N-N-N setup]")
print(f"WindFlow ONNX:          {safe_max(wf_onnx_thr)}")
print(f"WindFlow TF-SavedModel: {safe_max(wf_tfsm_thr)}")
print(f"WindFlow TF-Serving:    {safe_max(wf_tfsrv_thr)}")
print(f"WindFlow TorchServe:    {safe_max(wf_torchsrv_thr)}")
print("-" * 45)
print(f"Flink ONNX:             {safe_max(flink_onnx_thr)}")
print(f"Flink TF-SavedModel:    {safe_max(flink_tfsm_thr)}")
print(f"Flink TF-Serving:       {safe_max(flink_tfsrv_thr)}")
print(f"Flink TorchServe:       {safe_max(flink_torchsrv_thr)}")

print("\nMax Throughput per input rate (req/s)")
for i, r in enumerate(input_rates):
    print(
        f"ir ={r:>4}\n"
        f"   WF onnx = {wf_onnx_thr[i]} | tf-sm = {wf_tfsm_thr[i]} | tf-srv = {wf_tfsrv_thr[i]} | tsrv = {wf_torchsrv_thr[i]}\n"
        f"   FL onnx = {flink_onnx_thr[i]} | tf-sm = {flink_tfsm_thr[i]} | tf-srv = {flink_tfsrv_thr[i]} | tsrv = {flink_torchsrv_thr[i]}"
    )

### Vertical Scalability
Plot throughput for increasing parallelism

#### WindFlow

In [ ]:
import itertools
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl

# --- Config base ---
par_labels = ['2', '4', '8', '16', '32']
sp_key = "windflow"

# --- Label esperimenti (scalabilità) ---
scl_ffnn = ["30000_1_2", "30000_1_4", "30000_1_8", "30000_1_16", "30000_1_32"]
scl_res  = ["256_1_2", "256_1_4", "256_1_8", "256_1_16", "256_1_32"]

# --- Estrazione throughput da final_measurements ---
def get_metrics(labels, model_name):
    data = {"ONNX": [], "SavedModel": [], "TF-Serving": [], "TorchServe": []}
    for sc in labels:
        m_data = final_measurements.get(sc, {}).get(sp_key, {}).get(model_name, {})
        data["ONNX"].append(m_data.get("onnx", {}).get("throughput", 0))
        data["SavedModel"].append(m_data.get("tf-savedmodel", {}).get("throughput", 0))
        data["TF-Serving"].append(m_data.get("tf-serving", {}).get("throughput", 0))
        data["TorchServe"].append(m_data.get("torchserve", {}).get("throughput", 0))
    return data

data_ffnn = get_metrics(scl_ffnn, "ffnn")
data_res  = get_metrics(scl_res, "resnet50")

# --- Tabella numerica (stampa) ---
def print_table(title, par_list, metrics):
    print(f"\n{'='*65}")
    print(f" {title}")
    print(f"{'='*65}")
    header = f"{'Format':<15} | " + " | ".join([f"P={p:<6}" for p in par_list])
    print(header)
    print("-" * len(header))
    for fmt, values in metrics.items():
        row = f"{fmt:<15} | " + " | ".join([f"{v:>8.2f}" if v else f"{'N/A':>8}" for v in values])
        print(row)

print_table("FFNN THROUGHPUT (ir = 30k req/s)", par_labels, data_ffnn)
print_table("RESNET50 THROUGHPUT (ir = 256 req/s)", par_labels, data_res)

# --- Stile plot ---
mpl.rcParams['lines.linewidth'] = 4
matplotlib.rc('font', **{'size': 12})

tiffany, tiffany_dark = "#5BBDFF", "#4DFF79"
yellow, yellow_dark = "#FFD53E", "#FF3A3A"

# --- Layout figure (2 pannelli) ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4), gridspec_kw={'width_ratios': [1, 1]})

width = 0.2
offsets = [-(1.5 * width), -(0.5 * width), (0.5 * width), (1.5 * width)]
X = np.arange(len(par_labels))

# --- Plot a 4 barre (formati) ---
def plot_bars(ax, data, title, show_ylabel=False):
    hatches = itertools.cycle(['/', '+', '//', '-', 'x', '\\', '*', 'o'])
    formats = [("ONNX", tiffany), ("SavedModel", tiffany_dark), ("TF-Serving", yellow), ("TorchServe", yellow_dark)]

    for i, (fmt, color) in enumerate(formats):
        ax.bar(X + offsets[i], data[fmt], width=width, label=fmt,
               color=color, hatch=next(hatches), edgecolor='black')

    ax.set_xticks(X)
    ax.set_xticklabels(par_labels)
    ax.set_xlabel('Scoring Parallelism')
    if show_ylabel:
        ax.set_ylabel('Throughput (req/s)')
    ax.set_title(title)
    ax.grid(True, axis='y', linestyle='--', alpha=0.6)
    ax.set_axisbelow(True)

plot_bars(ax1, data_ffnn, 'FFNN (ir = 30k req/s)', show_ylabel=True)
plot_bars(ax2, data_res, 'ResNet50 (ir = 256 req/s)')

# --- Legenda + titolo + salvataggio ---
handles, labels = ax1.get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=4)

fig.suptitle("\nWindFlow Throughput Scalability", fontsize=16, y=1.05, fontweight='bold')

plt.tight_layout()
plt.savefig('scalability_results.pdf', bbox_inches='tight')
plt.show()

# Comparison between stream processors

## Sustainable throughput

### Latency under sustainable throughput

In [ ]:
import glob
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl

# --- Stile plot ---
mpl.rcParams['lines.linewidth'] = 4
mpl.rcParams['lines.markersize'] = 16
matplotlib.rc('font', **{'size': 12})

# --- Parametri ---
model = "ffnn"
trim_fraction = 0.25
input_rates = [4096]

series = [
    ("windflow", "WindFlow", "#18C4B0", "-"),
    ("flink", "Flink", "#D32F2F", "-"),
]

backends = [
    ("onnx", "ONNX"),
    ("tf-savedmodel", "TF-SavedModel"),
    ("tf-serving", "TF-Serving"),
    ("torchserve", "TorchServe")
]

# --- Output tabellare ---
print(f"\n{'='*95}")
print(f" FFNN LATENCY STATS (ir = {input_rates[0]} req/s) - All Backends")
print(f"{'='*95}")
print(f"{'Backend':<15} | {'SP':<10} | {'Mean':<10} | {'p50':<10} | {'p95':<10}")
print("-" * 95)

# --- Figura (2x2) ---
fig, axes = plt.subplots(2, 2, figsize=(9, 10), sharey=True)
axes_flat = axes.flatten()

# Limite X comune (per confrontabilità visiva)
GLOBAL_X_RIGHT = 800_000

for i, (backend, backend_title) in enumerate(backends):
    ax = axes_flat[i]
    ir = input_rates[0]
    pattern = f"*-ir{ir}-bs1-mr1.csv"

    for sp_key, label, color, ls in series:
        # File risultati (prende l'ultimo disponibile)
        files = sorted(glob.glob(f"../results/{sp_key}/{model}/{backend}/{pattern}"))
        if not files:
            continue

        picked = files[-1]

        # Parse CSV: (start,end)
        tt = []
        with open(picked, "r") as f_in:
            for line in f_in:
                parts = line.strip().split(",")
                if len(parts) < 2:
                    continue
                try:
                    tt.append((int(parts[0]), int(parts[1])))
                except ValueError:
                    continue

        tt.sort(key=lambda x: x[0])

        # Latenze (ms) con filtro > 1 ms
        times = [(end - start) for start, end in tt if (end - start) > 1]

        # Trim warm-up
        sz = int(trim_fraction * len(times))
        times = times[sz:]

        # Skip serie con pochi punti
        if len(times) < 10:
            continue

        # Statistiche (post-trim)
        mean = np.mean(times)
        p50  = np.percentile(times, 50)
        p95  = np.percentile(times, 95)
        print(f"{backend_title:<15} | {sp_key:<10} | {mean:>8.2f} | {p50:>8.2f} | {p95:>8.2f}")

        # CDF (steps-post) + plateau fino al limite globale
        sorted_times = np.sort(times)
        y_values = np.linspace(0, 1, len(sorted_times))

        if sorted_times[-1] < GLOBAL_X_RIGHT:
            sorted_times = np.append(sorted_times, GLOBAL_X_RIGHT)
            y_values = np.append(y_values, 1.0)

        ax.plot(
            sorted_times,
            y_values,
            label=label,
            color=color,
            linestyle=ls,
            drawstyle="steps-post"
        )

    # Assi e griglia
    ax.set_ylim(0, 1.05)
    ax.set_xlim(left=1000, right=GLOBAL_X_RIGHT)
    ax.grid(True, which="both", ls="-", alpha=0.5)
    ax.set_title(f'FFNN - {backend_title}', fontsize=12)

    if i >= 2:
        ax.set_xlabel('Latency (ms)')
    if i % 2 == 0:
        ax.set_ylabel('Cumulative Probability')

# Legenda + titolo + salvataggio
axes_flat[0].legend(loc='lower right', frameon=True, fontsize=10)
fig.suptitle(f'\nLatency CDF - FFNN (ir = {input_rates[0]} req/s)', fontsize=16, y=1.02, fontweight='bold')

plt.tight_layout()
plt.savefig('latency_cdf_4_backends_narrow.pdf', bbox_inches='tight')
plt.show()

In [ ]:
import glob
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl

# --- Stile plot ---
mpl.rcParams['lines.linewidth'] = 4
mpl.rcParams['lines.markersize'] = 16
matplotlib.rc('font', **{'size': 12})

# --- Parametri ---
model = "resnet50"
input_rates = [128]
trim_fraction = 0.25

series_config = [
    ("windflow", "WindFlow", "#18C4B0", "-"),
    ("flink", "Flink", "#D32F2F", "-"),
]

backends = [
    ("onnx", "ONNX"),
    ("tf-savedmodel", "TF-SavedModel"),
    ("tf-serving", "TF-Serving"),
    ("torchserve", "TorchServe")
]

# --- Output tabellare ---
print(f"\n{'='*95}")
print(f" RESNET50 LATENCY STATS (ir = {input_rates[0]} req/s) - All Backends")
print(f"{'='*95}")
print(f"{'Backend':<15} | {'SP':<10} | {'Mean':<10} | {'p50':<10} | {'p95':<10}")
print("-" * 95)

# --- Figura (2x2) ---
fig, axes = plt.subplots(2, 2, figsize=(9, 10), sharey=True)
axes_flat = axes.flatten()

for i, (backend_key, backend_title) in enumerate(backends):
    ax = axes_flat[i]
    ir = input_rates[0]
    pattern = f"*-ir{ir}-bs1-mr1.csv"

    all_times_in_panel = []
    per_series_times = []

    # --- Caricamento latenze (post-trim) per serie ---
    for sp_key, label, color, ls in series_config:
        files = sorted(glob.glob(f"../results/{sp_key}/{model}/{backend_key}/{pattern}"))

        times = []
        if files:
            picked = files[-1]
            tt = []
            with open(picked, "r") as f_in:
                for line in f_in:
                    parts = line.strip().split(",")
                    if len(parts) < 2:
                        continue
                    try:
                        tt.append((int(parts[0]), int(parts[1])))
                    except ValueError:
                        continue

            tt.sort(key=lambda x: x[0])
            times = [(end - start) for start, end in tt if (end - start) > 1]

            # trim warm-up
            sz = int(trim_fraction * len(times))
            times = times[sz:]

        if not times:
            continue

        all_times_in_panel.extend(times)
        per_series_times.append((sp_key, label, color, ls, times))

        # stampa stats
        print(
            f"{backend_title:<15} | {sp_key:<10} | {np.mean(times):>8.2f} | "
            f"{np.percentile(times, 50):>8.2f} | {np.percentile(times, 95):>8.2f}"
        )

    # --- Assi (limiti dinamici per pannello) ---
    ax.set_ylim(0, 1.05)

    x_right = None
    if all_times_in_panel:
        xmin = 10 ** np.floor(np.log10(np.min(all_times_in_panel)))
        xmax = 10 ** np.ceil(np.log10(np.max(all_times_in_panel)))
        ax.set_xlim(left=max(10, xmin), right=xmax)
        x_right = xmax
    else:
        # pannello vuoto
        ax.grid(True, which="both", ls="-", alpha=0.5)
        ax.set_title(f'{backend_title}', fontsize=12)
        if i >= 2:
            ax.set_xlabel('Latency (ms)')
        if i % 2 == 0:
            ax.set_ylabel('Cumulative Probability')
        continue

    # --- CDF (steps-post) + plateau fino a x_right ---
    for sp_key, label, color, ls, times in per_series_times:
        sorted_times = np.sort(times)
        y_values = np.linspace(0, 1, len(sorted_times))

        if x_right is not None and sorted_times[-1] < x_right:
            sorted_times = np.append(sorted_times, x_right)
            y_values = np.append(y_values, 1.0)

        ax.plot(
            sorted_times,
            y_values,
            label=label,
            color=color,
            linestyle=ls,
            drawstyle="steps-post"
        )

    ax.grid(True, which="both", ls="-", alpha=0.5)
    ax.set_title(f'ResNet50 - {backend_title}', fontsize=12)

    if i >= 2:
        ax.set_xlabel('Latency (ms)')
    if i % 2 == 0:
        ax.set_ylabel('Cumulative Probability')

# --- Legenda + titolo + export ---
axes_flat[0].legend(loc='lower right', frameon=True, fontsize=10)
fig.suptitle(f'\nLatency CDF - ResNet50 (ir = {input_rates[0]} req/s)', fontsize=16, y=1.02, fontweight='bold')

plt.tight_layout()
plt.subplots_adjust(bottom=0.15, wspace=0.12)

out_name = 'resnet50-latency-cdf-4panels-narrow.pdf'
plt.savefig(out_name, bbox_inches='tight')
plt.show()

### Vertical Scalability

In [ ]:
import itertools
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl

# --- Stile plot ---
mpl.rcParams['lines.linewidth'] = 4
mpl.rcParams['lines.markersize'] = 16
matplotlib.rc('font', **{'size': 14})

wf_color = "#18C4B0"
fl_color = "#D32F2F"
parallelism_labels = ['2', '4', '8', '16', '32']
formats = ["onnx", "tf-savedmodel", "tf-serving", "torchserve"]

# --- Fallback: dati sintetici se manca final_measurements ---
def get_data_mock(model_name, labels):
    results = {
        "windflow": {fmt: np.random.randint(100, 10000, 5).tolist() for fmt in formats},
        "flink":    {fmt: np.random.randint(100, 8000, 5).tolist() for fmt in formats}
    }
    return results

try:
    # Label esperimenti
    ffnn_labels = ["30000_1_2", "30000_1_4", "30000_1_8", "30000_1_16", "30000_1_32"]
    resnet_labels = ["256_1_2", "256_1_4", "256_1_8", "256_1_16", "256_1_32"]

    # --- Estrazione throughput da final_measurements ---
    def get_data(labels, model_name):
        results = {"windflow": {fmt: [] for fmt in formats}, "flink": {fmt: [] for fmt in formats}}
        for i, label in enumerate(labels):
            exps = final_measurements.get(label, {})
            for fmt in formats:
                wf_val = exps.get("windflow", {}).get(model_name, {}).get(fmt, {}).get("throughput", 0)
                results["windflow"][fmt].append(wf_val)

                fl_val = exps.get("flink", {}).get(model_name, {}).get(fmt, {}).get("throughput", 0)
                results["flink"][fmt].append(fl_val)
        return results

    ffnn_data = get_data(ffnn_labels, "ffnn")
    resnet_data = get_data(resnet_labels, "resnet50")
except NameError:
    print("ATTENZIONE: 'final_measurements' non trovato. Uso dati dummy.")
    ffnn_data = get_data_mock("ffnn", [])
    resnet_data = get_data_mock("resnet50", [])

# --- Plot: 4 pannelli (uno per backend) ---
def plot_4_panels(data, model_title, ylim=None, is_log=False):
    fig, axs = plt.subplots(1, 4, figsize=(16, 4), sharey=True)

    X = np.arange(len(parallelism_labels))
    width = 0.35

    wf_bar_color = wf_color
    fl_bar_color = fl_color
    hatches_list = ['/', '+', 'x', '\\', '*', 'o']

    for i, fmt in enumerate(formats):
        current_hatch = hatches_list[i % len(hatches_list)]

        axs[i].bar(
            X - width/2, data["windflow"][fmt], width=width, label='windflow',
            color=wf_bar_color, hatch=current_hatch, edgecolor='black'
        )
        axs[i].bar(
            X + width/2, data["flink"][fmt], width=width, label='flink',
            color=fl_bar_color, hatch=current_hatch, edgecolor='black'
        )

        axs[i].set_title(f'{model_title} - {fmt}', fontsize=15, pad=12)
        axs[i].set_xticks(X)
        axs[i].set_xticklabels(parallelism_labels)
        axs[i].set_xlabel('Scoring Parallelism')
        axs[i].grid(True, axis='y', linestyle='--', alpha=0.7)
        axs[i].set_axisbelow(True)

        if is_log:
            axs[i].set_yscale('log')

    axs[0].set_ylabel('Throughput (req/s)')

    if ylim:
        for ax in axs:
            ax.set_ylim(ylim)

    # Legenda unica (da primo pannello)
    handles, labels = axs[0].get_legend_handles_labels()
    unique_handles = handles[:2]
    unique_labels = labels[:2]
    fig.legend(
        unique_handles, unique_labels,
        loc='upper center', bbox_to_anchor=(0.5, 0.05),
        ncol=2, fontsize=14, frameon=False
    )

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.22, wspace=0.15)
    return fig

# --- Output FFNN ---
fig_ffnn = plot_4_panels(ffnn_data, "FFNN", ylim=[0, 13000])
plt.savefig('ffnn-scalability-4panels-wide.pdf', bbox_inches='tight')

# --- Output ResNet50 ---
fig_res = plot_4_panels(resnet_data, "ResNet50", ylim=[0, 45], is_log=False)
plt.savefig('resnet50-scalability-4panels-wide.pdf', bbox_inches='tight')

plt.show()

# Parallelism Models

In [ ]:
import itertools
import math
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.legend_handler import HandlerTuple

# --- Stile plot ---
mpl.rcParams['lines.linewidth'] = 4
mpl.rcParams['lines.markersize'] = 16
matplotlib.rc('font', **{'size': 12})

# --- Config ---
ir = ['2', '4', '8', '16', '32']
scl_labels = ["30000_1_2", "30000_1_4", "30000_1_8", "30000_1_16", "30000_1_32"]

flink_version  = "flink-32-N-32"
flink_matching = "flink"
wf_version     = "windflow-32-N-32"
wf_matching    = "windflow"

# --- Accumulatori throughput (ONNX) ---
sc_flink_onnx = []
sc_basef_onnx = []
sc_wf_onnx = []
sc_basew_onnx = []

# --- Accumulatori throughput (TF-Serving) ---
sc_flink_tf_serving = []
sc_basef_tf_serving = []
sc_wf_tf_serving = []
sc_basew_tf_serving = []

# --- Estrazione dati da final_measurements (fallback=1 se manca) ---
for sc in scl_labels:
    sc_exps = final_measurements.get(sc, {})

    # Flink 32-N-32
    if flink_version in sc_exps and "ffnn" in sc_exps[flink_version]:
        sc_flink_onnx.append(sc_exps[flink_version]["ffnn"].get("onnx", {}).get("throughput", 1))
        sc_flink_tf_serving.append(sc_exps[flink_version]["ffnn"].get("tf-serving", {}).get("throughput", 1))
    else:
        sc_flink_onnx.append(1)
        sc_flink_tf_serving.append(1)

    # Flink baseline
    if flink_matching in sc_exps and "ffnn" in sc_exps[flink_matching]:
        sc_basef_onnx.append(sc_exps[flink_matching]["ffnn"].get("onnx", {}).get("throughput", 1))
        sc_basef_tf_serving.append(sc_exps[flink_matching]["ffnn"].get("tf-serving", {}).get("throughput", 1))
    else:
        sc_basef_onnx.append(1)
        sc_basef_tf_serving.append(1)

    # WindFlow 32-N-32
    if wf_version in sc_exps and "ffnn" in sc_exps[wf_version]:
        sc_wf_onnx.append(sc_exps[wf_version]["ffnn"].get("onnx", {}).get("throughput", 1))
        sc_wf_tf_serving.append(sc_exps[wf_version]["ffnn"].get("tf-serving", {}).get("throughput", 1))
    else:
        sc_wf_onnx.append(1)
        sc_wf_tf_serving.append(1)

    # WindFlow baseline
    if wf_matching in sc_exps and "ffnn" in sc_exps[wf_matching]:
        sc_basew_onnx.append(sc_exps[wf_matching]["ffnn"].get("onnx", {}).get("throughput", 1))
        sc_basew_tf_serving.append(sc_exps[wf_matching]["ffnn"].get("tf-serving", {}).get("throughput", 1))
    else:
        sc_basew_onnx.append(1)
        sc_basew_tf_serving.append(1)

# --- Stampa tabelle ---
def print_row(label, values):
    vals_str = " | ".join([f"{v:>8.1f}" for v in values])
    print(f"{label:<25} | {vals_str}")

def print_section(title, base_wf, new_wf, base_fl, new_fl):
    print(f"\n{'='*80}")
    print(f" {title}")
    print(f"{'='*80}")
    header = f"{'System':<25} | " + " | ".join([f"P={p:<6}" for p in ir])
    print(header)
    print("-" * len(header))
    print_row("WindFlow [N-N-N]", base_wf)
    print_row("WindFlow [32-N-32]", new_wf)
    print("-" * len(header))
    print_row("Flink [N-N-N]", base_fl)
    print_row("Flink [32-N-32]", new_fl)
    print("=" * 80)

print_section("ONNX Throughput (req/s)", sc_basew_onnx, sc_wf_onnx, sc_basef_onnx, sc_flink_onnx)
print_section("TF-SERVING Throughput (req/s)", sc_basew_tf_serving, sc_wf_tf_serving, sc_basef_tf_serving, sc_flink_tf_serving)

# --- Differenze % (new vs baseline) ---
def pct_diff_str(new, baseline):
    if baseline <= 0:
        return "n/a"
    change = (new - baseline) / baseline * 100
    return f"{'+' if change >= 0 else ''}{math.floor(change)}%"

diffsf_onnx = [pct_diff_str(sc_flink_onnx[i], sc_basef_onnx[i]) for i in range(len(ir))]
diffsw_onnx = [pct_diff_str(sc_wf_onnx[i], sc_basew_onnx[i]) for i in range(len(ir))]

diffsf_tfserving = [pct_diff_str(sc_flink_tf_serving[i], sc_basef_tf_serving[i]) for i in range(len(ir))]
diffsw_tfserving = [pct_diff_str(sc_wf_tf_serving[i], sc_basew_tf_serving[i]) for i in range(len(ir))]

# --- Plot (2 pannelli verticali) ---
fig, [ax1, ax2] = plt.subplots(2, 1, figsize=(10, 12), sharey=True)

step = 1.35
X = np.arange(len(ir)) * step
group_span = 0.92
width = group_span / 4
gap = 0.04
offs = np.array([
    -(1.5 * width + gap / 2),
    -(0.5 * width + gap / 2),
     (0.5 * width + gap / 2),
     (1.5 * width + gap / 2),
])

wf_base_color = "#18C4B0"
wf_new_color  = "#129182"
fl_base_color = "#D32F2F"
fl_new_color  = "#972020"
hatches = itertools.cycle(['/', '+', '//', '-', 'x', '\\', '*', 'o', 'O', '.'])

# ONNX
ax1.bar(X + offs[0], sc_basew_onnx, width=width, label='windflow [N-N-N]',
        color=wf_base_color, hatch=next(hatches), edgecolor='black')
ax1.bar(X + offs[1], sc_wf_onnx, width=width, label='windflow [32-N-32]',
        color=wf_new_color, hatch=next(hatches), edgecolor='black')
ax1.bar(X + offs[2], sc_basef_onnx, width=width, label='flink [N-N-N]',
        color=fl_base_color, hatch=next(hatches), edgecolor='black')
ax1.bar(X + offs[3], sc_flink_onnx, width=width, label='flink [32-N-32]',
        color=fl_new_color, hatch=next(hatches), edgecolor='black')
ax1.set_title('FFNN - onnx', fontsize=14, pad=10)

# TF-Serving
hatches = itertools.cycle(['/', '+', '//', '-', 'x', '\\', '*', 'o', 'O', '.'])
ax2.bar(X + offs[0], sc_basew_tf_serving, width=width, label='windflow [N-N-N]',
        color=wf_base_color, hatch=next(hatches), edgecolor='black')
ax2.bar(X + offs[1], sc_wf_tf_serving, width=width, label='windflow [32-N-32]',
        color=wf_new_color, hatch=next(hatches), edgecolor='black')
ax2.bar(X + offs[2], sc_basef_tf_serving, width=width, label='flink [N-N-N]',
        color=fl_base_color, hatch=next(hatches), edgecolor='black')
ax2.bar(X + offs[3], sc_flink_tf_serving, width=width, label='flink [32-N-32]',
        color=fl_new_color, hatch=next(hatches), edgecolor='black')
ax2.set_title('FFNN - tf-serving', fontsize=14, pad=10)

# --- Formattazione comune ---
for ax in (ax1, ax2):
    ax.axhline(y=30000, color='r', linestyle='--', linewidth=2)
    ax.set_axisbelow(True)
    ax.grid(True)
    ax.set_xticks(X, ir)
    ax.set_ylabel('Throughput (req/s)')

ax1.set_ylim((0, 12500))
ax1.set_xlabel('')
ax2.set_xlabel('Scoring Parallelism')

# --- Annotazioni % ---
LABEL_SIZE = 15
TEXT_OFFSET = 400

for i in range(len(ir)):
    ax1.text(X[i] + offs[1], sc_wf_onnx[i] + TEXT_OFFSET, diffsw_onnx[i],
             ha='center', va='bottom', fontsize=LABEL_SIZE)
    ax1.text(X[i] + offs[3], sc_flink_onnx[i] + TEXT_OFFSET, diffsf_onnx[i],
             ha='center', va='bottom', fontsize=LABEL_SIZE)

    ax2.text(X[i] + offs[1], sc_wf_tf_serving[i] + TEXT_OFFSET, diffsw_tfserving[i],
             ha='center', va='bottom', fontsize=LABEL_SIZE)
    ax2.text(X[i] + offs[3], sc_flink_tf_serving[i] + TEXT_OFFSET, diffsf_tfserving[i],
             ha='center', va='bottom', fontsize=LABEL_SIZE)

# --- Legenda combinata (4 voci) ---
handles, labels_ = ax1.get_legend_handles_labels()
handles2, labels2_ = ax2.get_legend_handles_labels()
unique_labels = ['windflow [N-N-N]', 'windflow [32-N-32]', 'flink [N-N-N]', 'flink [32-N-32]']
labels_all = labels_ + labels2_
handles_all = handles + handles2
combined_handles = [tuple([h for h, l in zip(handles_all, labels_all) if l == label]) for label in unique_labels]

ax2.legend(
    handles=combined_handles,
    labels=unique_labels,
    handlelength=3,
    handler_map={tuple: HandlerTuple(ndivide=None, pad=0)},
    ncol=4,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.25)
)

plt.subplots_adjust(hspace=0.3, bottom=0.15)
plt.savefig('parallelism-models-vertical-serving-12k.pdf', bbox_inches='tight', pad_inches=0.15)
plt.show()

### Bursty Workloads Analysis

In [ ]:
import csv
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib

# --- Stile plot ---
mpl.rcParams['lines.linewidth'] = 1.5
matplotlib.rc('font', **{'size': 11})

# --- File CSV (run) ---
wf_runs_2500 = [
    "../results/windflow/ffnn/onnx/20260217014630838-ir2500-bs1-mr1.csv",
    "../results/windflow/ffnn/onnx/20260217004920466-ir2500-bs1-mr1.csv"
]

wf_runs_1300 = [
    "../results/windflow/ffnn/onnx/20260130160809895-ir1300-bs1-mr1.csv",
    "../results/windflow/ffnn/onnx/20260211165941254-ir1300-bs1-mr1.csv"
]

fl_runs_1300 = [
    "../results/flink/ffnn/onnx/20260130163833580-ir1300-bs1-mr1.csv",
    "../results/flink/ffnn/onnx/20260211204626042-ir1300-bs1-mr1.csv"
]

# --- Parametri generali ---
bd, tbb = 30, 120
TS_UNIT = "ms"
start_from_record = 1000
pre_sec, post_sec = 20, 20
mio_offset = 0
RUN2_SHIFT = 2

# --- Shift globale (allineamento) ---
GLOBAL_SHIFT = 7

# --- Finestra di visualizzazione ---
VIEW_START = 110
VIEW_END = 540

# --- Pattern burst (definito a mano) ---
PATTERN_START = 140.0
PLATEAU_WIDTH = 42.0
BURST_INTERVAL = 148.0
NUM_BURSTS = 3

# --- Metriche recovery ---
RECOVERY_THRESHOLD_MS = 500.0
STABLE_WINDOW_SEC = 5.0
MAX_SEARCH_AFTER_BURST_SEC = 200.0

# --- Colori ---
wf_colors = ["#18C4B0", "#6EEDDE"]
fl_colors = ["#D32F2F", "#FFB3B3"]

# --- Utility tempo ---
def ts_to_sec(x):
    return x / 1e3 if TS_UNIT == "ms" else x / 1e6

# --- Carica CSV e costruisce (t, lat) ---
def build_data(csv_path, start_from_record=0):
    starts, ends = [], []
    try:
        with open(csv_path, "r") as f:
            r = csv.reader(f)
            for row in r:
                if len(row) < 2:
                    continue
                try:
                    starts.append(int(row[0]))
                    ends.append(int(row[1]))
                except ValueError:
                    continue
    except FileNotFoundError:
        print(f"File not found: {csv_path}")
        return np.array([]), np.array([])

    s, e = np.array(starts), np.array(ends)
    if len(s) == 0:
        return np.array([]), np.array([])

    idx = np.argsort(s)
    s, e = s[idx], e[idx]

    t_abs = ts_to_sec(s - s[0])
    lat = (e - s) if TS_UNIT == "ms" else (e - s) / 1e3

    if len(t_abs) > start_from_record:
        return t_abs[start_from_record:], lat[start_from_record:]
    return t_abs, lat

# --- Crop finestra e riallinea a t_start ---
def crop_and_align(t, lat, t_start, t_end):
    mask = (t >= t_start) & (t <= t_end)
    return t[mask] - t_start, lat[mask]

# --- Finestre burst ---
def build_burst_windows():
    windows = []
    for i in range(NUM_BURSTS):
        s = PATTERN_START + (i * BURST_INTERVAL)
        e = s + PLATEAU_WIDTH
        windows.append((s, e))
    return windows

# --- Recovery: primo istante con finestra stabile sotto soglia ---
def recovery_time_after_burst(t, lat_ms, burst_end, thr_ms, stable_sec, max_search_sec):
    if t.size == 0:
        return None, None
    start_t = burst_end
    end_t = burst_end + max_search_sec

    mask = (t >= start_t) & (t <= end_t)
    tt = t[mask]
    ll = lat_ms[mask]
    if tt.size == 0:
        return None, None

    for i in range(tt.size):
        t_i = tt[i]
        t_target = t_i + stable_sec
        j = np.searchsorted(tt, t_target, side="left")
        if j <= i:
            continue
        if j > tt.size:
            break
        if np.max(ll[i:j]) <= thr_ms:
            return float(t_i), float(t_i - burst_end)

    return None, None

# --- AUC sopra soglia dopo burst ---
def auc_recovery(t, lat_ms, burst_end, thr_ms, max_search_sec):
    if t.size == 0:
        return None
    start_t = burst_end
    end_t = burst_end + max_search_sec

    mask = (t >= start_t) & (t <= end_t)
    tt = t[mask]
    ll = lat_ms[mask]
    if tt.size < 2:
        return None

    above = np.maximum(0.0, ll - thr_ms)
    return float(np.trapz(above, tt))

# --- Carica serie (2 run) e applica crop ---
def load_series(paths, start_from_record, t0, t1_big):
    series = []
    for run_idx in [0, 1]:
        t, l = build_data(paths[run_idx], start_from_record)
        tt, ll = crop_and_align(t, l, t0, t1_big)
        series.append((tt, ll))
    return series

# --- Stampa metriche recovery per burst ---
def compute_recovery_metrics(series, name, run_shift_sec=0.0, global_offset=0.0):
    print("\n" + "=" * 90)
    print(f"RECOVERY METRICS - {name}")
    print(f"Global Shift applied: {global_offset}s")
    print("=" * 90)

    header = "Run | Burst | burst_end(s) | T_rec(s) | Δ_rec(s) | AUC_above_thr(ms·s)"
    print(header)
    print("-" * len(header))

    for run_idx in [0, 1]:
        t, lat = series[run_idx]
        shift = run_shift_sec if run_idx == 1 else 0.0
        t_plot = t + shift + global_offset

        for b_idx, (bs, be) in enumerate(burst_windows, start=1):
            t_rec, d_rec = recovery_time_after_burst(
                t_plot, lat,
                burst_end=be,
                thr_ms=RECOVERY_THRESHOLD_MS,
                stable_sec=STABLE_WINDOW_SEC,
                max_search_sec=MAX_SEARCH_AFTER_BURST_SEC
            )
            auc = auc_recovery(
                t_plot, lat,
                burst_end=be,
                thr_ms=RECOVERY_THRESHOLD_MS,
                max_search_sec=MAX_SEARCH_AFTER_BURST_SEC
            )

            be_str = f"{be:10.1f}"
            if t_rec is None:
                t_rec_str = f">{(be + MAX_SEARCH_AFTER_BURST_SEC):.1f}"
                d_rec_str = f">{MAX_SEARCH_AFTER_BURST_SEC:.1f}"
            else:
                t_rec_str = f"{t_rec:7.1f}"
                d_rec_str = f"{d_rec:7.1f}"

            auc_str = "NA" if auc is None else f"{auc:18.1f}"
            print(f"{run_idx+1:>3} | {b_idx:>5} | {be_str} | {t_rec_str:>7} | {d_rec_str:>7} | {auc_str}")

# --- Plot serie (2 run) con shift ---
def plot_series(ax, series, colors, title, run2_shift, global_shift):
    for run_idx in [1, 0]:
        shift = run2_shift if run_idx == 1 else 0
        z_order = 1 if run_idx == 1 else 10
        t, lat = series[run_idx]
        ax.plot(t + shift + global_shift, lat,
                color=colors[run_idx], label=f"Run {run_idx+1}", zorder=z_order)

    ax.set_title(title)
    h, l = ax.get_legend_handles_labels()
    ax.legend(h[::-1], l[::-1], loc="upper right")

# --- Caricamento e crop ---
t0 = mio_offset * tbb - pre_sec
t1_big = 10_000

wf_2500_series = load_series(wf_runs_2500, start_from_record, t0, t1_big)
wf_1300_series = load_series(wf_runs_1300, start_from_record, t0, t1_big)
fl_1300_series = load_series(fl_runs_1300, start_from_record, t0, t1_big)

burst_windows = build_burst_windows()

# --- Metriche recovery (stampa) ---
compute_recovery_metrics(wf_2500_series, "WindFlow @ 2500 req/s", run_shift_sec=RUN2_SHIFT, global_offset=GLOBAL_SHIFT)
compute_recovery_metrics(wf_1300_series, "WindFlow @ 1300 req/s", run_shift_sec=RUN2_SHIFT, global_offset=GLOBAL_SHIFT)
compute_recovery_metrics(fl_1300_series, "Flink @ 1300 req/s",   run_shift_sec=RUN2_SHIFT, global_offset=GLOBAL_SHIFT)

# --- Figura (4 pannelli) ---
fig, (ax0, ax1, ax_mid, ax2) = plt.subplots(4, 1, figsize=(12, 13), sharex=True)

# 1) Pattern target (sintetico)
tt = np.linspace(VIEW_START, VIEW_END, 2000)
yy = np.full_like(tt, -0.3)
for i in range(NUM_BURSTS):
    s = PATTERN_START + (i * BURST_INTERVAL)
    e = s + PLATEAU_WIDTH
    yy[(tt >= s) & (tt <= e)] = 0.1

ax0.plot(tt, yy, color="black", linewidth=2)
ax0.axhline(0.0, color="red", linestyle="--", linewidth=1.5)
ax0.text(VIEW_START + (VIEW_END - VIEW_START) * 0.2, 0.02, "Sustainable Throughput", color="red")
ax0.set_title("Target Throughput Pattern")
ax0.set_ylim(-0.4, 0.4)
ax0.set_yticks([-0.4, -0.2, 0.0, 0.2, 0.4])
ax0.set_ylabel("events/sec")

# Linee burst e soglia recovery (solo grafici latenza)
for (bs, be) in burst_windows:
    for ax in (ax1, ax_mid, ax2):
        ax.axvline(be, color="black", alpha=0.15, linestyle="--")
for ax in (ax1, ax_mid, ax2):
    ax.axhline(RECOVERY_THRESHOLD_MS, color="black", alpha=0.15, linestyle=":")

# 2) WF 2500
plot_series(ax1, wf_2500_series, wf_colors, "WindFlow @ 2500 req/s", RUN2_SHIFT, GLOBAL_SHIFT)

# 3) WF 1300
plot_series(ax_mid, wf_1300_series, wf_colors, "WindFlow @ 1300 req/s", RUN2_SHIFT, GLOBAL_SHIFT)

# 4) Flink 1300
plot_series(ax2, fl_1300_series, fl_colors, "Flink @ 1300 req/s", RUN2_SHIFT, GLOBAL_SHIFT)

# --- Formattazione ---
for ax in [ax0, ax1, ax_mid, ax2]:
    ax.grid(True, alpha=0.4, linestyle=':')
    ax.set_xlim(VIEW_START, VIEW_END)
    if ax != ax0:
        ax.set_ylabel("latency (ms)")
ax2.set_xlabel("Time (sec)")

# --- Scala Y comune (solo limite max) ---
YMAX = 23000
ymin = min(ax1.get_ylim()[0], ax_mid.get_ylim()[0], ax2.get_ylim()[0])
ax1.set_ylim(ymin, YMAX)
ax_mid.set_ylim(ymin, YMAX)
ax2.set_ylim(ymin, YMAX)

plt.tight_layout()
plt.show()